# Mood Music Agent Demo

상황을 직접 입력하면 `mood_music_agent.py`가 Claude와 ReccoBeats를 사용해 곡을 추천합니다.

- 추천 로직은 이 노트북에 새로 만들지 않고 `mood_music_agent.recommend_music()`만 호출합니다.
- 감정 점수는 선택 입력입니다. 비워두면 상황 문장만으로 감정 방향을 추론합니다.
- 선호사항은 선택 입력입니다. 장르 선호는 seed 유도와 후보 AI 검증 필터에 쓰고, 제외 아티스트는 후처리 필터로 제거합니다.
- 실행 시 Anthropic API와 ReccoBeats API를 실제 호출합니다.
- `.env`에 Anthropic API key가 있어야 합니다.

In [ ]:
import json
from pprint import pprint

import mood_music_agent as agent

print("Loaded mood_music_agent from:", agent.__file__)

## 1. 상황 입력

아래 셀을 실행하면 상황을 입력할 수 있습니다. 예시는 다음과 같습니다.

- `비 오는 밤에 혼자 집까지 걸어가면서 하루를 정리하고 싶다`
- `친구들과 한강에서 피크닉 중이고 너무 들뜨지는 않는 밝은 음악이 필요하다`
- `새벽에 과제 마감 직전이라 불안하지만 집중해야 한다`

In [ ]:
situation = input("지금 상황을 입력하세요: ").strip()

if not situation:
    raise ValueError("상황을 한 문장 이상 입력해 주세요.")

situation

## 2. 감정 점수 선택 입력

감정 점수는 선택 사항입니다. 그냥 Enter를 누르면 `{}`로 실행합니다.

입력하려면 JSON 형식으로 넣으세요.

예: `{"sadness": 0.7, "tired": 0.8, "calm": 0.4}`

In [ ]:
import json

emotion_text = input("감정 점수 JSON, 없으면 Enter: ").strip()

if emotion_text:
    emotions = json.loads(emotion_text)
else:
    emotions = {}

if not isinstance(emotions, dict):
    raise ValueError("감정 점수는 JSON object 형식이어야 합니다.")

emotions

## 3. 선호사항 선택 입력

선호사항도 선택 사항입니다. 그냥 Enter를 누르면 `{}`로 실행합니다.

사용 가능한 키:

- `preferred_genres`: 선호 장르. Claude seed 유도와 추천 후보 AI 검증 필터에 반영합니다.
- `avoid_genres`: 피하고 싶은 장르. Claude seed 방향과 추천 후보 AI 검증 필터에 반영합니다.
- `preferred_artists`: 선호 아티스트. Claude seed 유도와 최종 랭킹 boost에 반영합니다.
- `avoid_artists`: 제외 아티스트. Claude seed에서 피하고, 최종 추천 결과에서도 제거합니다.

예: `{"preferred_genres": ["Korean R&B", "city pop"], "preferred_artists": ["DEAN", "wave to earth"], "avoid_genres": ["metal"], "avoid_artists": ["Artist Name"]}`

In [ ]:
import mood_music_agent as agent

preferences_text = input("선호사항 JSON, 없으면 Enter: ").strip()
preferences = agent.parse_preferences_text(preferences_text)
preferences

## 4. 추천 개수 설정

In [ ]:
import mood_music_agent as agent

if "situation" not in globals():
    raise ValueError("먼저 1번 상황 입력 셀을 실행해 주세요.")
if "emotions" not in globals():
    emotions = {}

limit_text = input("추천 개수, 기본 5: ").strip()
limit = int(limit_text) if limit_text else 5

agent.validate_inputs(situation, emotions, limit)
limit

## 5. 추천 실행

이 셀은 실제 API를 호출합니다. 결과에는 ReccoBeats에 전달한 `valence`, `danceability`, `energy`, `tempo`, `popularity` target과 적용된 preferences도 함께 표시됩니다.

In [ ]:
from pprint import pprint

import mood_music_agent as agent

if "situation" not in globals():
    raise ValueError("먼저 1번 상황 입력 셀을 실행해 주세요.")
if "emotions" not in globals():
    emotions = {}
if "preferences" not in globals():
    preferences = {}
if "limit" not in globals():
    limit = 5

result = agent.recommend_music(situation, emotions, limit=limit, preferences=preferences)

summary = {
    "situation": situation,
    "emotions": emotions,
    "preferences": preferences,
    "mood_label": result["mood_profile"]["mood_label"],
    "listening_intent": result["mood_profile"]["listening_intent"],
    "reccobeats_target_params": result["sources"]["reccobeats_target_params"],
    "warnings": result["warnings"],
}

pprint(summary, sort_dicts=False)

## 6. 추천곡 확인

In [ ]:
from pprint import pprint

if "result" not in globals():
    raise ValueError("먼저 5번 추천 실행 셀을 실행해 주세요.")

rows = []

for rank, item in enumerate(result["recommendations"], start=1):
    rows.append({
        "rank": rank,
        "title": item["title"],
        "artists": ", ".join(item["artists"]),
        "popularity": item["popularity"],
        "valence": item["audio_features"].get("valence"),
        "danceability": item["audio_features"].get("danceability"),
        "energy": item["audio_features"].get("energy"),
        "tempo": item["audio_features"].get("tempo"),
        "spotify_url": item["spotify_url"],
        "fit_reason": item["fit_reason"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    pprint(rows, sort_dicts=False)

## 7. 원본 JSON 확인

In [ ]:
import json

if "result" not in globals():
    raise ValueError("먼저 5번 추천 실행 셀을 실행해 주세요.")

print(json.dumps(result, ensure_ascii=False, indent=2))